# Mode-classification calibration on GS2 linear databases (Fusion_PhD-7k70)

Calibrates the cutoffs of `general_analysis.mode_classification` on three GS2 Latin-hypercube databases:
NSTX_MTM n1000 (MTM-rich), SPR-045 and M1 kbm_8d (KBM-rich). Three indicators per case, from the final time of
each run: tearing parameter `T` (Hatch 2012), signed frequency `omega` (ion direction > 0, pyrokinetics
convention) and `chi_e/chi_i` (Kotschenreuther 2019), plus `D_e/chi_e`. Consensus: a label only if every
indicator agrees, otherwise `mixed`. All indicators are computed here from the pyro objects; nothing is cached.

In [ ]:
import os, warnings
from multiprocessing import Pool
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
from pyrokinetics import Pyro
from general_analysis import mode_classification as mc

warnings.filterwarnings("ignore")
load_dotenv(Path.cwd().parent / "local.env")
data_root = Path(os.environ["GK_DATA_ROOT"])
code_, run_template, project = "GS2", "Runs", "LATIN_HYPERCUBE"
# database label: (case, scan_information)
databases = {"NSTX_MTM": ("NSTX_MTM", "beta_q_shat_ky_n1000"), "SPR-045": ("SPR-045", "kbm_8d"), "M1": ("M1", "kbm_8d")}
n_spot, seed = 5, 0
plot_dir = Path.cwd().parent / "Plots" / "mode_classification"
plt.style.use(Path(mc.__file__).parent / "paper.mplstyle")

In [ ]:
def run(args):
    db, d = args
    try:
        pyro = Pyro(gk_file=d / "gs2.in")
        pyro.load_gk_output(load_fields=True, load_fluxes=True)
        gamma = float(pyro.gk_output["growth_rate"].isel(time=-1).squeeze().values)
        return dict(db=db, run=d.name, growth_rate=gamma, ky=float(pyro.numerics.ky.m), **mc.indicators(pyro))
    except Exception as e:
        return dict(db=db, run=d.name, err=repr(e)[:120])

jobs = [(db, d) for db, (case, scan) in databases.items()
        for d in sorted((data_root / code_ / run_template / project / case / scan).glob("iteration_*"))]
with Pool(int(os.environ.get("SLURM_CPUS_PER_TASK", 4))) as pool:
    raw = pd.DataFrame(pool.map(run, jobs, chunksize=4))

In [ ]:
# Nothing is dropped silently: load failures, then non-positive growth rate, then non-finite indicators.
failed = raw[raw.get("err").notna()] if "err" in raw else raw.iloc[:0]
ok = raw.drop(failed.index)
print("loaded", raw.groupby("db").size().to_dict(), "| load failures", failed.groupby("db").size().to_dict())
stable = ok[ok.growth_rate <= 0]
ok = ok[ok.growth_rate > 0]
nonfinite = ok[~np.isfinite(ok[["T", "omega", "chi_ratio", "de_chi"]]).all(axis=1)]
df = ok.drop(nonfinite.index).copy()
print("growth_rate <= 0", stable.groupby("db").size().to_dict(), "| non-finite indicator (zero flux)", nonfinite.groupby("db").size().to_dict())
print("kept", df.groupby("db").size().to_dict())

## Sign convention check
pyrokinetics states `mode_frequency < 0` is the electron direction (comments in its CGYRO and GENE readers). Check on the
unambiguous populations: NSTX_MTM cases with tearing parity and huge `chi_e/chi_i` are MTMs and must be electron direction;
SPR-045 + M1 cases with ballooning parity, `chi_e/chi_i` ~ 1 and `D_e/chi_e` ~ 2/3 are the KBM population.

In [ ]:
mtm_like = df[(df["T"] > 0.7) & (df.chi_ratio > 30)]
kbm_like = df[(df["T"] < 0.05) & df.chi_ratio.between(0.5, 2) & df.de_chi.between(0.5, 1) & (df.db != "NSTX_MTM")]
print(f"MTM-like (T>0.7, chi_e/chi_i>30): {len(mtm_like)} cases, omega<0 (electron) in {(mtm_like.omega < 0).mean():.3f}")
print(f"KBM-like (T<0.05, chi 0.5-2, De/chie 0.5-1): {len(kbm_like)} cases, omega>0 (ion) in {(kbm_like.omega > 0).mean():.3f}")

## Indicator distributions and pairwise scatters (cutoffs from the module drawn as lines)

In [ ]:
col = {"NSTX_MTM": "tab:red", "SPR-045": "tab:blue", "M1": "tab:green"}
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4))
bins = {"T": np.logspace(-6, 0, 40), "omega": np.linspace(-5, 5, 50), "chi_ratio": np.logspace(-2, 3.5, 40), "de_chi": np.linspace(-3, 3, 50)}
for a, (k, b) in zip(ax, bins.items()):
    for db, g in df.groupby("db"):
        a.hist(np.clip(g[k], b[0], b[-1]), bins=b, histtype="step", label=db, color=col[db])
    a.set_xlabel(k)
    if k in ("T", "chi_ratio"): a.set_xscale("log")
ax[0].axvline(mc.T_BAL, c="k", ls="--"); ax[0].axvline(mc.T_TEAR, c="k", ls=":")
ax[1].axvline(0, c="k", ls="--")
ax[2].axvspan(*mc.CHI_KBM, color="grey", alpha=0.2); ax[2].axvline(mc.CHI_MTM, c="k", ls=":")
ax[3].axvline(mc.DE_CHI_KBM, c="k", ls="--")
ax[0].legend()
fig.savefig(plot_dir / "distributions.png")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8.5))
pairs = [("T", "chi_ratio"), ("omega", "chi_ratio"), ("omega", "T"), ("de_chi", "chi_ratio"), ("de_chi", "T"), ("omega", "de_chi")]
for a, (x, y) in zip(ax.flat, pairs):
    for db, g in df.groupby("db"):
        a.scatter(g[x], g[y], s=6, alpha=0.5, label=db, c=col[db])
    a.set_xlabel(x); a.set_ylabel(y)
    if x == "T": a.set_xscale("symlog", linthresh=1e-3)
    if y == "chi_ratio": a.set_yscale("symlog", linthresh=0.1); a.axhspan(*mc.CHI_KBM, color="grey", alpha=0.15); a.axhline(mc.CHI_MTM, c="k", ls=":")
    if y == "T": a.set_yscale("symlog", linthresh=1e-3); a.axhline(mc.T_BAL, c="k", ls="--"); a.axhline(mc.T_TEAR, c="k", ls=":")
    if x == "T": a.axvline(mc.T_BAL, c="k", ls="--"); a.axvline(mc.T_TEAR, c="k", ls=":")
    if x == "omega": a.axvline(0, c="k", ls="--")
    if y == "de_chi": a.set_ylim(-3, 3)
    if x == "de_chi": a.set_xlim(-3, 3)
ax[0, 0].legend()
fig.savefig(plot_dir / "pairwise_scatter.png")

## Consensus labels, and how many cases each single indicator would label differently
Single-indicator labels: `omega` > 0 is KBM-like, < 0 MTM-like; `T` < T_BAL KBM-like, > T_TEAR MTM-like;
`chi_e/chi_i` in the KBM band KBM-like, above the MTM cut MTM-like; `D_e/chi_e` > cutoff KBM-like (no MTM label).
"Differs" counts cases the single indicator labels KBM or MTM-like where the consensus label is different.

In [ ]:
df["label"] = [mc.classify(r) for r in df.to_dict("records")]
frac = df.groupby("db").label.value_counts().unstack(fill_value=0)
print(frac.assign(total=frac.sum(axis=1)))
print((frac.div(frac.sum(axis=1), axis=0)).round(3))
single = pd.DataFrame({
    "omega": np.where(df.omega > 0, "KBM", "MTM"),
    "T": np.select([df["T"] < mc.T_BAL, df["T"] > mc.T_TEAR], ["KBM", "MTM"], "mixed"),
    "chi": np.select([df.chi_ratio.between(*mc.CHI_KBM), df.chi_ratio > mc.CHI_MTM], ["KBM", "MTM"], "mixed"),
    "de_chi": np.where(df.de_chi > mc.DE_CHI_KBM, "KBM", "mixed")})
rows = {k: df.assign(s=v).groupby("db").apply(lambda g: ((g.s != "mixed") & (g.s != g.label)).sum()) for k, v in single.items()}
print("cases where the single indicator gives KBM/MTM but the consensus label differs:")
print(pd.DataFrame(rows))
print("consensus KBM cases that would be 'mixed' without the D_e/chi_e condition (the ITG/TEM separator):")
no_de = pd.Series([mc.classify({**r, "de_chi": np.inf}) for r in df.to_dict("records")], index=df.index)
print(((no_de == "KBM") & (df.label != "KBM")).groupby(df.db).sum())

## Comparison with Kotschenreuther 2019, Table 1
Table 1 (A): MHD-like (KBM) chi_i/chi_e ~ 1, D_e/chi_e ~ 2/3; MTM chi_i/chi_e ~ 1/10, D_e/chi_e ~ 1/10. (B): ITG/TEM chi_e/chi_i 1/4 to 1.
Shown as chi_e/chi_i (inverse of the table's chi_i/chi_e). Boxes are the 25-75% range of our labelled cases.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ref = {"KBM": (1, 2 / 3), "MTM": (10, 0.1)}
for k, (name, y) in enumerate({"chi_ratio": "chi_e/chi_i", "de_chi": "D_e/chi_e"}.items()):
    groups = [df[df.label == l][name] for l in ("KBM", "MTM", "mixed")]
    ax[k].boxplot([g.clip(-50, 500) for g in groups], tick_labels=[f"{l}\n(n={len(g)})" for l, g in zip(("KBM", "MTM", "mixed"), groups)], showfliers=False)
    ax[k].scatter([1, 2], [ref["KBM"][k], ref["MTM"][k]], marker="*", s=200, c="gold", edgecolor="k", label="Kotschenreuther T1", zorder=5)
    ax[k].set_ylabel(y)
    if k == 0: ax[k].set_yscale("log")
ax[0].legend()
fig.savefig(plot_dir / "kotschenreuther_comparison.png")
print(df.groupby("label")[["chi_ratio", "de_chi", "T", "omega"]].quantile([.25, .5, .75]).round(3))

## Spot check: 5 random labelled cases per class against GS2 |A_par| and |phi|
Seeded random draw (seed above). Eigenfunctions are from the same gk_output, normalised to their own maximum.

In [ ]:
rng = np.random.default_rng(seed)
for lab in ("KBM", "MTM", "mixed"):
    pick = df[df.label == lab]
    pick = pick.iloc[rng.choice(len(pick), min(n_spot, len(pick)), replace=False)]
    fig, ax = plt.subplots(2, len(pick), figsize=(3.2 * len(pick), 5), squeeze=False)
    for j, r in enumerate(pick.itertuples()):
        case, scan = databases[r.db]
        d = data_root / code_ / run_template / project / case / scan / r.run
        pyro = Pyro(gk_file=d / "gs2.in"); pyro.load_gk_output(load_fields=True, load_fluxes=True)
        for i, f in enumerate(("apar", "phi")):
            v = pyro.gk_output[f].isel(time=-1).squeeze()
            ax[i, j].plot(v.theta, np.abs(v) / np.abs(v).max())
            ax[i, j].set_xlabel("theta")
        ax[0, j].set_title(f"{r.db} {r.run}\nT={r.T:.2f} w={r.omega:.2f}\nchi={r.chi_ratio:.2f} De/chie={r.de_chi:.2f}", fontsize=7)
    ax[0, 0].set_ylabel("|A_par|"); ax[1, 0].set_ylabel("|phi|")
    fig.suptitle(f"consensus label: {lab}")
    fig.savefig(plot_dir / f"spotcheck_{lab}.png")